# stratosampler — Examples

Covers the full workflow: loading molecules, stratified splitting, quality metrics, and visualisation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from stratosampler import (
    PropertyStratifiedSplitter,
    compute_properties,
    BUILTIN_PROPERTIES,
    load_smiles,
    distribution_report,
    split_summary,
    coverage_score,
    plot_property_distributions,
    plot_split_comparison,
    plot_chemical_space,
)

## 1. Sample dataset

We build a small in-memory DataFrame of SMILES strings to keep this notebook self-contained. Swap in your own data at any point.

In [ ]:
SMILES = [
    # lipophilic / large
    "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1",  # imatinib
    "COc1ccc2nccc(OC)c2c1",
    "CC(C)(C)c1ccc(NC(=O)Nc2ccc(Cl)c(Cl)c2)cc1",
    "O=C(Nc1ccc(F)cc1)c1cnc2ccccc2c1",
    "Cc1cc(NC2=NC(=O)c3ccccc32)ccc1Cl",
    # polar / small
    "CC(=O)Nc1ccc(O)cc1",   # paracetamol
    "OC(=O)c1ccccc1O",      # salicylic acid
    "Nc1ccc(S(=O)(=O)N)cc1",
    "CC(O)=O",              # acetic acid
    "NCC(=O)O",             # glycine
    "OC[C@H](O)[C@@H](O)[C@H](O)CO",  # sorbitol
    "c1ccc(N)cc1",          # aniline
    "OC(=O)CCC(=O)O",
    # medium range
    "CCc1ccc(S(=O)(=O)N)cc1",
    "Cc1ccc(C(=O)O)cc1",
    "FC(F)(F)c1ccc(NC(=O)c2ccco2)cc1",
    "O=C(O)c1ccc(F)cc1",
    "Clc1ccccc1NC(=O)c1cccs1",
    "COc1ccc(C(=O)O)cc1",
    "c1ccc2[nH]ccc2c1",     # indole
    "O=c1[nH]cc[nH]1",
    "CCc1ccc(O)cc1",
    "CC(C)Cc1ccc(C(C)C(=O)O)cc1",   # ibuprofen
    "CC1(C)CC(=O)c2ccccc21",
    "O=C(O)CC(=O)O",        # malonic acid
    "OC(=O)c1ccc(O)cc1",
    "Cc1ccc(Cl)cc1",
    "CC(C)c1ccc(O)cc1",
    "CCCCC(=O)O",
    "CCc1ccc(Cl)cc1",
]

df = pd.DataFrame({"smiles": SMILES})
print(f"{len(df)} compounds")

## 2. Compute physicochemical properties

In [ ]:
print("Available built-in properties:", list(BUILTIN_PROPERTIES.keys()))

In [ ]:
PROPS = ["MolLogP", "MolWt", "TPSA", "NumHDonors", "NumHAcceptors"]

prop_df = compute_properties(df["smiles"], PROPS)
df = pd.concat([df, prop_df], axis=1)
df.head()

In [ ]:
df[PROPS].describe().round(2)

## 3. Basic train / test split

In [ ]:
splitter = PropertyStratifiedSplitter(
    properties=PROPS,
    n_bins=4,
    test_size=0.2,
    random_state=42,
)

train_idx, test_idx = splitter.split(df, smiles_col="smiles")
print(f"Train: {len(train_idx)}  Test: {len(test_idx)}")

### Distribution quality report

In [ ]:
report = distribution_report(df, train_idx, test_idx, PROPS)
report.round(3)

`ks_stat` close to 0 and `js_divergence` close to 0 mean train and test follow similar distributions.

### Property distribution plots

In [ ]:
fig = plot_property_distributions(df, train_idx, test_idx, PROPS, n_bins=10)
plt.tight_layout()
plt.show()

## 4. Train / validation / test split

In [ ]:
splitter3 = PropertyStratifiedSplitter(
    properties=["MolLogP", "MolWt", "TPSA"],
    n_bins=4,
    test_size=0.15,
    val_size=0.15,
    random_state=42,
)

train_idx3, val_idx3, test_idx3 = splitter3.split(df, smiles_col="smiles")
print(f"Train: {len(train_idx3)}  Val: {len(val_idx3)}  Test: {len(test_idx3)}")

In [ ]:
report3 = distribution_report(df, train_idx3, test_idx3, PROPS, val_idx=val_idx3)
report3.round(3)

In [ ]:
fig3 = plot_property_distributions(
    df, train_idx3, test_idx3, PROPS, val_idx=val_idx3, n_bins=10
)
plt.tight_layout()
plt.show()

## 5. get_split_dataframes — work directly with DataFrames

In [ ]:
train_df, test_df = splitter.get_split_dataframes(df, smiles_col="smiles")
print(f"train_df: {train_df.shape}  test_df: {test_df.shape}")
train_df.head()

## 6. Scaffold-aware splitting

With `scaffold_aware=True`, molecules sharing the same Murcko scaffold are kept together in the same split, preventing data leakage between structurally similar compounds.

In [ ]:
splitter_scaffold = PropertyStratifiedSplitter(
    properties=["MolLogP", "MolWt"],
    n_bins=4,
    test_size=0.2,
    scaffold_aware=True,
    random_state=42,
)

train_sc, test_sc = splitter_scaffold.split(df, smiles_col="smiles")
print(f"Train: {len(train_sc)}  Test: {len(test_sc)}")

## 7. Using pre-computed property columns

If properties are already in the DataFrame, pass them via `property_cols` to skip RDKit computation.

In [ ]:
splitter_cols = PropertyStratifiedSplitter(
    properties=["MolLogP", "MolWt", "TPSA"],
    n_bins=4,
    test_size=0.2,
    random_state=0,
)

train_c, test_c = splitter_cols.split(df, property_cols=["MolLogP", "MolWt", "TPSA"])
print(f"Train: {len(train_c)}  Test: {len(test_c)}")

## 8. Stratified vs random split — quality comparison

In [ ]:
rng = np.random.default_rng(42)
n = len(df)
perm = rng.permutation(n)
cut = int(n * 0.8)
rand_train = perm[:cut]
rand_test = perm[cut:]

summary_random = split_summary(df, rand_train, rand_test, PROPS)
summary_strat = split_summary(df, train_idx, test_idx, PROPS)

print(f"Random   — mean KS: {summary_random['mean_ks_stat']:.3f}  mean JS: {summary_random['mean_js_div']:.3f}")
print(f"Stratified— mean KS: {summary_strat['mean_ks_stat']:.3f}  mean JS: {summary_strat['mean_js_div']:.3f}")

In [ ]:
fig_cmp = plot_split_comparison(
    {"random": summary_random, "stratified": summary_strat},
    PROPS,
    metric="ks_stat",
)
plt.tight_layout()
plt.show()

## 9. Chemical space visualisation

In [ ]:
fig_cs = plot_chemical_space(df, train_idx, test_idx, x_col="MolLogP", y_col="MolWt")
plt.tight_layout()
plt.show()

## 10. Coverage score

Fraction of strata that appear in both train and test — 1.0 is ideal.

In [ ]:
from stratosampler import coverage_score
from stratosampler.splitters.property_stratified import compute_properties
from sklearn.preprocessing import KBinsDiscretizer

prop_vals = df[["MolLogP", "MolWt"]].fillna(0).values
kbd = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")
bins = kbd.fit_transform(prop_vals).astype(int)
strata = [f"{r[0]}_{r[1]}" for r in bins]

score = coverage_score(train_idx, test_idx, strata)
print(f"Coverage score: {score:.2f}")

## 11. Loading molecules from files

These cells assume you have a SMILES file or SDF handy. Adjust the paths accordingly.

In [ ]:
# --- SMILES file (one SMILES per line, optional ID column) ---
# mols, data = load_smiles("compounds.smi")

# --- CSV with SMILES in column 1 and IDs in column 0 ---
# mols, data = load_smiles(
#     "compounds.csv",
#     delimiter=",",
#     smiles_column=1,
#     id_column=0,
#     keep_properties=True,
# )

# --- SDF file ---
# from stratosampler import load_sdf
# mols, data = load_sdf("compounds.sdf")

print("Uncomment the block that matches your file format.")

## 12. Full pipeline in one block

In [ ]:
# 1. data already loaded into `df` with a 'smiles' column
splitter_full = PropertyStratifiedSplitter(
    properties=["MolLogP", "MolWt", "TPSA"],
    n_bins=4,
    test_size=0.2,
    val_size=0.1,
    scaffold_aware=False,
    random_state=42,
)

# 2. split
tr, va, te = splitter_full.split(df, smiles_col="smiles")

# 3. quality
summary = split_summary(df, tr, te, ["MolLogP", "MolWt", "TPSA"], val_idx=va)
print(f"n_train={summary['n_train']}  n_val={summary['n_val']}  n_test={summary['n_test']}")
print(f"mean KS stat: {summary['mean_ks_stat']:.3f}")
print(f"mean JS div:  {summary['mean_js_div']:.3f}")

# 4. export
train_out = df.iloc[tr].reset_index(drop=True)
val_out   = df.iloc[va].reset_index(drop=True)
test_out  = df.iloc[te].reset_index(drop=True)

# train_out.to_csv("train.csv", index=False)
# val_out.to_csv("val.csv", index=False)
# test_out.to_csv("test.csv", index=False)
print("Done.")